# Spatial Diagnostics — Milton (N=21 counties)

**Objective**: Diagnose whether nonlinear relationships between socioeconomic variables
and mobility disruption/recovery metrics are spatial in nature.

**DVs** (within-region only at county level):
- `largest_drop_within` — max % reduction from SARIMAX baseline
- `recovery_within` — trend-based recovery days
- `outflow_increase` — max % increase above baseline (descriptive)

**IVs**: total_population, median_household_income, pct_no_vehicle, insurance_coverage_pct

**Steps**:
1. Load data & compute county centroid → storm track distance
2. Scatter + LOWESS for each DV × IV, colored by distance-to-track
3. OLS regression + Moran's I on residuals
4. Interpretation & next steps

In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns

import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.preprocessing import StandardScaler

from libpysal.weights import Queen, KNN, DistanceBand
from esda.moran import Moran

from shapely.geometry import LineString
from shapely.ops import nearest_points

warnings.filterwarnings("ignore", category=FutureWarning)

# Output directory
OUTPUT_DIR = "../results/spatial_diagnostics_milton/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "figures"), exist_ok=True)
print("Setup complete.")

Setup complete.


/Users/qing/miniconda3/envs/geo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data

In [5]:
# ── Load Milton DVs ──
results_dir = "../results/milton/"

drop_df = pd.read_csv(os.path.join(results_dir, "largest_drop_within.csv"))
recovery_df = pd.read_csv(os.path.join(results_dir, "recovery_within.csv"))
outflow_df = pd.read_csv(os.path.join(results_dir, "outflow_increase.csv"))

# Merge all DVs
dvs = drop_df[["GEOID", "largest_drop"]].merge(
    recovery_df[["GEOID", "recovery_days"]], on="GEOID", how="outer"
).merge(
    outflow_df[["GEOID", "largest_increase"]], on="GEOID", how="outer"
)
dvs["GEOID"] = dvs["GEOID"].astype(int)

print(f"Milton DVs loaded: {len(dvs)} counties")
display(dvs.describe().round(2))

Milton DVs loaded: 21 counties


,GEOID,largest_drop,recovery_days,largest_increase
count,21.00,21.00,21.00,21.00
mean,12077.10,-35.03,4.87,61.10
std,34.57,6.51,0.49,51.38
min,12009.00,-47.24,4.14,9.39
25%,12055.00,-40.78,4.55,26.53
50%,12081.00,-35.21,4.80,40.18
75%,12103.00,-30.99,5.10,76.45
max,12127.00,-23.53,6.22,177.83


In [6]:
# ── Load ACS socioeconomic data ──
acs_df = pd.read_csv("acs_socioeconomic_v2.csv")
acs_df["GEOID"] = acs_df["GEOID"].astype(int)

FEATURES = [
    "total_population",
    "median_household_income",
    "pct_no_vehicle",
    "insurance_coverage_pct",
]

# Merge
df = dvs.merge(acs_df[["GEOID"] + FEATURES], on="GEOID", how="inner")
print(f"After merge with ACS: {len(df)} counties")
display(df.head())

After merge with ACS: 21 counties


,GEOID,largest_drop,recovery_days,largest_increase,total_population,median_household_income,pct_no_vehicle,insurance_coverage_pct
0,12009,-28.697301,5.036244,26.526502,610723,71308.0,4.666937,98.605778
1,12015,-47.237604,4.457580,177.825958,189900,62164.0,4.502132,99.023261
2,12027,-40.577553,4.143078,32.515765,34258,45000.0,6.199352,98.581973
3,12049,-25.778323,4.702109,23.629464,25528,44665.0,4.909561,98.687346
4,12053,-42.456910,5.338690,75.144807,196621,59202.0,5.065114,98.389251


In [7]:
# ── Load county names ──
geo_idx = pd.read_csv("geoid_idx_names.csv")
geo_idx["GEOID"] = geo_idx["GEOID"].astype(int)
df = df.merge(geo_idx[["GEOID", "NAME"]], on="GEOID", how="left")
print(f"Counties: {df['NAME'].tolist()}")

Counties: ['Brevard', 'Charlotte', 'DeSoto', 'Hardee', 'Hernando', 'Highlands', 'Hillsborough', 'Indian River', 'Lake', 'Lee', 'Manatee', 'Okeechobee', 'Orange', 'Osceola', 'Pasco', 'Pinellas', 'Polk', 'Sarasota', 'Seminole', 'Sumter', 'Volusia']


## 2. Compute Distance to Hurricane Track

In [8]:
# ── Load storm track ──
track_path = "./../../hurricane_oct/data/storm_track/milton_storm_track.shp"
track_gdf = gpd.read_file(track_path)
print(f"Track CRS: {track_gdf.crs}")
print(f"Track records: {len(track_gdf)}")
display(track_gdf.head())

Track CRS: EPSG:4326
Track records: 55


,sid,season,number,basin,subbasin,name,iso_time,nature,lat,lon,...,usa_sea_sw,usa_sea_nw,storm_spd,storm_dr,year,month,day,hour,min,geometry
0,2024278N21265,2024,71,NA,GM,MILTON,2024-10-04 00:00:00,DS,20.6,-94.7,...,NaN,NaN,2,295,2024,10,4,0,0,POINT (-94.7 20.6)
1,2024278N21265,2024,71,NA,GM,MILTON,2024-10-04 03:00:00,DS,20.7,-94.8,...,NaN,NaN,2,300,2024,10,4,3,0,POINT (-94.8 20.7)
2,2024278N21265,2024,71,NA,GM,MILTON,2024-10-04 06:00:00,DS,20.7,-94.9,...,NaN,NaN,2,305,2024,10,4,6,0,POINT (-94.9 20.7)
3,2024278N21265,2024,71,NA,GM,MILTON,2024-10-04 09:00:00,DS,20.7,-95.0,...,NaN,NaN,1,315,2024,10,4,9,0,POINT (-95 20.7)
4,2024278N21265,2024,71,NA,GM,MILTON,2024-10-04 12:00:00,DS,20.8,-95.0,...,NaN,NaN,1,320,2024,10,4,12,0,POINT (-95 20.8)


In [9]:
# ── Load county geometries ──
county_shp_path = "./../../hurricane_oct/data/county_geo/tl_2023_us_county/tl_2023_us_county.shp"
counties_gdf = gpd.read_file(county_shp_path)
counties_gdf["GEOID"] = counties_gdf["GEOID"].astype(int)

# Filter to Milton counties
milton_geoids = df["GEOID"].tolist()
counties_milton = counties_gdf[counties_gdf["GEOID"].isin(milton_geoids)].copy()
print(f"Milton county geometries: {len(counties_milton)}")

Milton county geometries: 21


In [10]:
# ── Project to Albers Equal Area (EPSG:5070) for distance in meters ──
counties_proj = counties_milton.to_crs(epsg=5070)
track_proj = track_gdf.to_crs(epsg=5070)

# Build a single LineString from the track points
if track_proj.geometry.geom_type.iloc[0] == "Point":
    track_line = LineString(track_proj.geometry.tolist())
elif track_proj.geometry.geom_type.iloc[0] == "LineString":
    track_line = track_proj.unary_union
else:
    # MultiLineString or mixed
    track_line = track_proj.unary_union

print(f"Track geometry type: {track_line.geom_type}")
print(f"Track length: {track_line.length / 1000:.1f} km")

Track geometry type: LineString
Track length: 2483.5 km


In [11]:
# ── Compute centroid → track distance (km) ──
counties_proj["centroid"] = counties_proj.geometry.centroid
counties_proj["dist_to_track_m"] = counties_proj["centroid"].apply(
    lambda pt: track_line.distance(pt)
)
counties_proj["dist_to_track_km"] = counties_proj["dist_to_track_m"] / 1000.0
counties_proj["dist_to_track_mi"] = counties_proj["dist_to_track_km"] / 1.60934

# Merge distance back to main df
dist_df = counties_proj[["GEOID", "dist_to_track_km", "dist_to_track_mi"]].copy()
dist_df["GEOID"] = dist_df["GEOID"].astype(int)
df = df.merge(dist_df, on="GEOID", how="left")

print("\nDistance to track summary (miles):")
print(df[["NAME", "dist_to_track_mi"]].sort_values("dist_to_track_mi").to_string(index=False))

# Save distance to CSV
df[["GEOID", "NAME", "dist_to_track_km", "dist_to_track_mi"]].to_csv(
    os.path.join(OUTPUT_DIR, "county_distance_to_track.csv"), index=False
)
print(f"\nSaved distance data to {OUTPUT_DIR}/county_distance_to_track.csv")


Distance to track summary (miles):
        NAME  dist_to_track_mi
        Polk          1.676145
     Manatee          2.942559
     Brevard         10.315948
    Sarasota         11.289462
     Osceola         12.587094
      Hardee         18.407624
      Orange         19.920764
Hillsborough         24.427360
    Seminole         29.942454
      DeSoto         34.944728
    Pinellas         41.477201
   Charlotte         43.319139
   Highlands         44.468651
        Lake         46.011896
     Volusia         48.803460
Indian River         50.843682
       Pasco         51.809095
      Sumter         56.823550
  Okeechobee         59.489780
         Lee         61.718577
    Hernando         66.109596

Saved distance data to ../results/spatial_diagnostics_milton//county_distance_to_track.csv


## 3. Scatter + LOWESS Plots Colored by Distance-to-Track

For each DV × IV pair: scatter plot with LOWESS smoother, points colored by distance to track.
This reveals (a) whether the relationship is nonlinear, and (b) whether the nonlinearity follows a spatial gradient.

In [12]:
DV_CONFIGS = {
    "largest_drop": {
        "label": "Largest Drop — Within (%)",
        "col": "largest_drop",
    },
    "recovery_days": {
        "label": "Recovery Time — Within (days)",
        "col": "recovery_days",
    },
    "largest_increase": {
        "label": "Outflow Increase (%)",
        "col": "largest_increase",
    },
}

IV_LABELS = {
    "total_population": "Total Population",
    "median_household_income": "Median Household Income ($)",
    "pct_no_vehicle": "% No Vehicle",
    "insurance_coverage_pct": "Insurance Coverage (%)",
    "dist_to_track_mi": "Distance to Track (mi)",
}

In [ ]:
def plot_scatter_lowess_distance(df, iv_col, dv_col, iv_label, dv_label,
                                  dist_col="dist_to_track_mi", save_path=None):
    """
    Scatter plot with LOWESS smoother, points colored by distance-to-track.
    Bubble size = population. Labels show NAME + GEOID.
    """
    fig, ax = plt.subplots(figsize=(10, 7))

    # Color by distance
    norm = mcolors.Normalize(vmin=df[dist_col].min(), vmax=df[dist_col].max())
    cmap = cm.RdYlGn_r  # red=close, green=far

    # Bubble size by population (scale to visible range)
    pop = df["total_population"].fillna(df["total_population"].median())
    size_min, size_max = 40, 400
    pop_norm = (pop - pop.min()) / (pop.max() - pop.min())
    sizes = size_min + pop_norm * (size_max - size_min)

    sc = ax.scatter(
        df[iv_col], df[dv_col],
        c=df[dist_col], cmap=cmap, norm=norm,
        s=sizes, edgecolors="black", linewidth=0.8, zorder=5, alpha=0.8
    )
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("Distance to Track (mi)", fontsize=11)

    # LOWESS smoother
    valid = df[[iv_col, dv_col]].dropna()
    if len(valid) >= 5:
        lowess_result = lowess(valid[dv_col], valid[iv_col], frac=0.6, it=3)
        ax.plot(lowess_result[:, 0], lowess_result[:, 1],
                color="blue", linewidth=2.5, label="LOWESS (frac=0.6)", zorder=4)

    # OLS fit line + p-value
    from scipy.stats import spearmanr, pearsonr
    r_p, p_p = pearsonr(valid[iv_col], valid[dv_col])
    r_s, p_s = spearmanr(valid[iv_col], valid[dv_col])
    z = np.polyfit(valid[iv_col], valid[dv_col], 1)
    x_range = np.linspace(valid[iv_col].min(), valid[iv_col].max(), 100)
    ax.plot(x_range, np.polyval(z, x_range),
            color="red", linewidth=1.5, linestyle="--",
            label=f"OLS (p={p_p:.3f})", zorder=3)

    # Annotate with NAME + GEOID
    # Highlight problematic inflow counties
    problem_geoids = [12009, 12055, 12061]
    for _, row in df.iterrows():
        if pd.notna(row[dv_col]):
            geoid = int(row["GEOID"])
            label = f"{row['NAME']}\n({geoid})"
            fontcolor = "red" if geoid in problem_geoids else "black"
            fontweight = "bold" if geoid in problem_geoids else "normal"
            ax.annotate(
                label,
                (row[iv_col], row[dv_col]),
                fontsize=6, alpha=0.8, color=fontcolor, fontweight=fontweight,
                xytext=(5, 5), textcoords="offset points"
            )

    # Stats box
    ax.text(0.02, 0.98,
            f"Pearson r={r_p:.3f} (p={p_p:.3f})\nSpearman ρ={r_s:.3f} (p={p_s:.3f})\nBubble size = population",
            transform=ax.transAxes, fontsize=9, va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="wheat", alpha=0.5))

    ax.set_xlabel(iv_label, fontsize=12)
    ax.set_ylabel(dv_label, fontsize=12)
    ax.set_title(f"{dv_label} vs {iv_label}", fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.show()

In [ ]:
# ── Generate all scatter + LOWESS plots ──
# IVs: socioeconomic + distance-to-track
all_ivs = FEATURES + ["dist_to_track_mi"]

for dv_key, dv_cfg in DV_CONFIGS.items():
    dv_col = dv_cfg["col"]
    dv_label = dv_cfg["label"]

    print(f"\n{'='*60}")
    print(f"DV: {dv_label}")
    print(f"{'='*60}")

    for iv_col in all_ivs:
        iv_label = IV_LABELS.get(iv_col, iv_col)
        save_path = os.path.join(
            OUTPUT_DIR, "figures",
            f"scatter_{dv_key}_vs_{iv_col}.png"
        )
        plot_scatter_lowess_distance(
            df, iv_col, dv_col, iv_label, dv_label,
            save_path=save_path
        )

## 4. OLS Regression + Moran's I on Residuals

For each DV:
1. Run OLS with standardized socioeconomic features
2. Extract residuals
3. Compute Moran's I using Queen contiguity weights → tests for spatial autocorrelation
4. Also test with KNN weights (k=4) as alternative

**Interpretation of Moran's I:**
- Significant positive I → nearby counties have similar residuals → spatial structure not captured by IVs
- Non-significant I → residuals are spatially random → nonlinearity is likely non-spatial

In [ ]:
# ── Prepare spatial weights ──
# Merge geometry back to df for spatial analysis
gdf = counties_proj[["GEOID", "geometry"]].merge(df, on="GEOID", how="inner")
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:5070")
gdf = gdf.sort_values("GEOID").reset_index(drop=True)

print(f"GeoDataFrame ready: {len(gdf)} counties")

# Queen contiguity weights
try:
    w_queen = Queen.from_dataframe(gdf)
    w_queen.transform = "r"  # row-standardized
    print(f"Queen weights: {w_queen.n} observations, mean neighbors = {w_queen.mean_neighbors:.1f}")
    # Check for islands (counties with no neighbors)
    islands = [i for i, v in w_queen.neighbors.items() if len(v) == 0]
    if islands:
        print(f"  WARNING: {len(islands)} island(s) detected: {gdf.iloc[islands]['NAME'].tolist()}")
except Exception as e:
    print(f"Queen weights failed: {e}")
    w_queen = None

# KNN weights (k=4) — more robust for irregular county shapes
w_knn = KNN.from_dataframe(gdf, k=4)
w_knn.transform = "r"
print(f"KNN(k=4) weights: {w_knn.n} observations")

In [ ]:
def run_ols_morans(gdf, dv_col, dv_label, features, w_queen, w_knn, output_dir):
    """
    Run OLS regression and compute Moran's I on residuals.
    
    Returns: OLS model, residuals, Moran's I results
    """
    # Drop NaN rows
    valid = gdf[[dv_col] + features + ["NAME", "dist_to_track_mi"]].dropna()
    valid_idx = valid.index
    
    if len(valid) < 5:
        print(f"  Skipping {dv_label}: only {len(valid)} valid observations")
        return None, None, None
    
    # Standardize IVs
    scaler = StandardScaler()
    X_raw = valid[features].values
    X_z = scaler.fit_transform(X_raw)
    X_z_df = pd.DataFrame(X_z, columns=features, index=valid.index)
    X = sm.add_constant(X_z_df)
    y = valid[dv_col]
    
    # Fit OLS
    model = sm.OLS(y, X).fit()
    
    print(f"\n{'='*60}")
    print(f"OLS: {dv_label} (N={len(valid)})")
    print(f"{'='*60}")
    print(f"R² = {model.rsquared:.4f}, Adj. R² = {model.rsquared_adj:.4f}")
    print(f"F-stat = {model.fvalue:.4f}, Prob(F) = {model.f_pvalue:.4e}")
    print("\nCoefficients:")
    coef_summary = pd.DataFrame({
        "coef": model.params,
        "std_err": model.bse,
        "t": model.tvalues,
        "p": model.pvalues,
    }).round(4)
    display(coef_summary)
    
    # Residuals
    residuals = model.resid
    
    # ── Moran's I on residuals ──
    print("\n--- Moran's I on OLS residuals ---")
    
    # Need to subset weights to match valid observations
    morans_results = {}
    
    for w_name, w in [("Queen", w_queen), ("KNN(k=4)", w_knn)]:
        if w is None:
            print(f"  {w_name}: skipped (weights not available)")
            continue
        
        try:
            mi = Moran(residuals.values, w)
            print(f"  {w_name}: I = {mi.I:.4f}, E[I] = {mi.EI:.4f}, "
                  f"z = {mi.z_norm:.4f}, p = {mi.p_norm:.4f}")
            sig = "***" if mi.p_norm < 0.01 else "**" if mi.p_norm < 0.05 else "*" if mi.p_norm < 0.1 else ""
            print(f"           → {'Significant' if mi.p_norm < 0.05 else 'Not significant'} "
                  f"spatial autocorrelation {sig}")
            morans_results[w_name] = mi
        except Exception as e:
            print(f"  {w_name}: failed — {e}")
    
    return model, residuals, morans_results

In [ ]:
# ── Run OLS + Moran's I for each DV ──
ols_results = {}

for dv_key, dv_cfg in DV_CONFIGS.items():
    dv_col = dv_cfg["col"]
    dv_label = dv_cfg["label"]
    
    model, residuals, morans = run_ols_morans(
        gdf, dv_col, dv_label, FEATURES, w_queen, w_knn, OUTPUT_DIR
    )
    ols_results[dv_key] = {
        "model": model,
        "residuals": residuals,
        "morans": morans,
    }

## 5. Residual Maps

Map OLS residuals to visually check for spatial clustering.

In [ ]:
def plot_residual_map(gdf, residuals, dv_label, save_path=None):
    """
    Choropleth map of OLS residuals.
    Blue = over-predicted (negative residual), Red = under-predicted (positive residual).
    """
    if residuals is None:
        return
    
    plot_gdf = gdf.copy()
    plot_gdf["residual"] = np.nan
    plot_gdf.loc[residuals.index, "residual"] = residuals.values
    plot_gdf = plot_gdf.dropna(subset=["residual"])
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    vmax = max(abs(plot_gdf["residual"].min()), abs(plot_gdf["residual"].max()))
    plot_gdf.plot(
        column="residual",
        cmap="RdBu_r",
        legend=True,
        legend_kwds={"label": "OLS Residual"},
        edgecolor="black",
        linewidth=0.8,
        ax=ax,
        vmin=-vmax,
        vmax=vmax,
    )
    
    # Annotate county names
    for _, row in plot_gdf.iterrows():
        centroid = row.geometry.centroid
        ax.annotate(
            row["NAME"],
            xy=(centroid.x, centroid.y),
            fontsize=7,
            ha="center",
            va="center",
            color="black",
            weight="bold",
        )
    
    ax.set_title(f"OLS Residuals: {dv_label}", fontsize=14)
    ax.set_axis_off()
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.show()

In [ ]:
for dv_key, dv_cfg in DV_CONFIGS.items():
    dv_label = dv_cfg["label"]
    res = ols_results[dv_key]
    save_path = os.path.join(OUTPUT_DIR, "figures", f"residual_map_{dv_key}.png")
    plot_residual_map(gdf, res["residuals"], dv_label, save_path=save_path)

## 6. Residuals vs Distance-to-Track

Direct test: do residuals correlate with distance to track?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (dv_key, dv_cfg) in zip(axes, DV_CONFIGS.items()):
    res = ols_results[dv_key]
    if res["residuals"] is None:
        continue
    
    residuals = res["residuals"]
    dist = gdf.loc[residuals.index, "dist_to_track_mi"]
    names = gdf.loc[residuals.index, "NAME"]
    
    ax.scatter(dist, residuals, s=80, edgecolors="black", linewidth=0.6, c="steelblue")
    ax.axhline(0, color="gray", ls="--", lw=1)
    
    # LOWESS
    if len(residuals) >= 5:
        lw = lowess(residuals.values, dist.values, frac=0.6)
        ax.plot(lw[:, 0], lw[:, 1], color="red", linewidth=2, label="LOWESS")
    
    # Annotate
    for name, d, r in zip(names, dist, residuals):
        ax.annotate(name, (d, r), fontsize=6, alpha=0.6, xytext=(3, 3),
                    textcoords="offset points")
    
    # Correlation
    from scipy.stats import spearmanr
    rho, p = spearmanr(dist, residuals)
    ax.set_title(f"{dv_cfg['label']}\nρ={rho:.3f}, p={p:.3f}", fontsize=11)
    ax.set_xlabel("Distance to Track (mi)")
    ax.set_ylabel("OLS Residual")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "figures", "residuals_vs_distance.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved: {save_path}")
plt.show()

## 7. Summary Table

Consolidate all diagnostic results.

In [ ]:
summary_rows = []

for dv_key, dv_cfg in DV_CONFIGS.items():
    res = ols_results[dv_key]
    model = res["model"]
    morans = res["morans"]
    
    if model is None:
        continue
    
    row = {
        "DV": dv_cfg["label"],
        "N": int(model.nobs),
        "R²": round(model.rsquared, 4),
        "Adj. R²": round(model.rsquared_adj, 4),
        "F-stat": round(model.fvalue, 4),
        "F p-value": f"{model.f_pvalue:.4e}",
    }
    
    # Add Moran's I results
    for w_name in ["Queen", "KNN(k=4)"]:
        if morans and w_name in morans:
            mi = morans[w_name]
            row[f"Moran I ({w_name})"] = round(mi.I, 4)
            row[f"Moran p ({w_name})"] = round(mi.p_norm, 4)
        else:
            row[f"Moran I ({w_name})"] = None
            row[f"Moran p ({w_name})"] = None
    
    # Residuals vs distance correlation
    residuals = res["residuals"]
    if residuals is not None:
        from scipy.stats import spearmanr
        dist = gdf.loc[residuals.index, "dist_to_track_mi"]
        rho, p = spearmanr(dist, residuals)
        row["Resid~Dist ρ"] = round(rho, 4)
        row["Resid~Dist p"] = round(p, 4)
    
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*80)
print("DIAGNOSTIC SUMMARY")
print("="*80)
display(summary_df)

# Save
summary_df.to_csv(os.path.join(OUTPUT_DIR, "diagnostic_summary.csv"), index=False)
print(f"\nSaved to {OUTPUT_DIR}/diagnostic_summary.csv")

## 8. Interpretation Guide

**Reading the results:**

| Result | Meaning | Next step |
|--------|---------|----------|
| OLS F p > 0.05 | Global linear model not significant | Nonlinear model needed |
| LOWESS curves from linear | Nonlinear relationship exists | GAM or threshold model |
| Moran's I p < 0.05 | Residuals spatially autocorrelated | Spatial model (interaction with distance) |
| Moran's I p > 0.05 | No spatial autocorrelation in residuals | Nonlinearity is non-spatial → GAM |
| Resid~Dist ρ significant | Residuals correlate with track distance | Distance is a missing predictor or interaction |
| Color gradient in scatter | Nonlinearity follows distance gradient | Distance × IV interaction model |

**Decision tree:**
1. If Moran's I NOT significant AND Resid~Dist NOT significant → **GAM** (nonlinear, non-spatial)
2. If Moran's I significant OR Resid~Dist significant → **Distance interaction model**: `y ~ β₁x + β₂d + β₃(x·d)`
3. If scatter color shows threshold → **Piecewise regression** at the distance cutoff